<!-- NB_DOC_INTRO_V1 -->
# Deep Learning — PyTorch Lightning

> 📚 **Doc thematique** : [docs/04_DL.md](docs/04_DL.md) (Deep Learning)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**PyTorch Lightning** = couche d'abstraction au-dessus de PyTorch qui supprime le boilerplate. Tu gardes : `nn.Module`, `optim`, `DataLoader`. Tu supprimes : boucle d'entrainement manuelle, `.to(device)`, `model.train()/eval()`, `loss.backward()`, `optimizer.step()`, gradient accumulation, mixed precision, distributed (DDP), checkpointing.

Ce notebook execute un classifieur **MNIST complet en Lightning** (CPU OK pour la demo, GPU si dispo) : `LightningModule`, `Trainer`, callbacks, loggers.

Versions : `lightning >= 2.2`, `torch >= 2.2`.

## Plan

1. Avant/apres Lightning (boilerplate compare)
2. Pattern `LightningModule`
3. **Demo MNIST executable** (CPU, 1 epoch pour rapidite)
4. Trainer et ses options (accelerateurs, precision)
5. Callbacks (EarlyStopping, ModelCheckpoint, LearningRateMonitor, SWA)
6. Loggers (TensorBoard, Wandb, MLflow)
7. Mixed precision (FP16, BF16)
8. Distributed training (DDP, FSDP, DeepSpeed)
9. LightningDataModule pour separer data et modele
10. Migration script PyTorch → Lightning (cote a cote)
11. Pieges et anti-patterns
12. References


## 1. Avant / Apres Lightning

### Avant (PyTorch pur — extrait simplifie)
```python
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MyModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        # val loop, log...
        ...
```

### Apres (Lightning)
```python
class MyModel(L.LightningModule):
    def __init__(self): ...
    def forward(self, x): ...
    def training_step(self, batch, _):
        x, y = batch
        return F.cross_entropy(self(x), y)
    def validation_step(self, batch, _):
        x, y = batch
        self.log("val_loss", F.cross_entropy(self(x), y))
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

trainer = L.Trainer(max_epochs=10, accelerator="auto")
trainer.fit(MyModel(), train_loader, val_loader)
```

→ ~30 lignes → 10 lignes. Et gratuitement : GPU auto, logging, checkpoints, early stop.


## 2. Demo MNIST complete (executable)

On utilise une **micro-archi** (MLP) sur un subset MNIST + 1 seul epoch pour aller vite en CPU (~ 30s).


In [ ]:
# pip install -q lightning torch torchvision
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # eviter conflit MKL sur certains setups

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
import lightning as L

# Charger torchvision uniquement si dispo
try:
    from torchvision import datasets, transforms
    TORCHVISION = True
except ImportError:
    TORCHVISION = False
    print("torchvision indisponible. Demo sur donnees synthetiques.")

print(f"Lightning : {L.__version__}")
print(f"Torch     : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

SEED = 42
L.seed_everything(SEED, workers=True)

In [ ]:
# === LightningModule ===
class MNISTClassifier(L.LightningModule):
    def __init__(self, hidden_dim: int = 64, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()   # log dans hparams + acces via self.hparams.lr
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(-1) == y).float().mean()
        self.log_dict({"train_loss": loss, "train_acc": acc},
                      prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(-1) == y).float().mean()
        self.log_dict({"val_loss": loss, "val_acc": acc}, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = (logits.argmax(-1) == y).float().mean()
        self.log("test_acc", acc)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

print("LightningModule defini.")

In [ ]:
# === Data : MNIST si dispo, sinon synthetique ===
import tempfile

if TORCHVISION:
    transform = transforms.ToTensor()
    data_dir = tempfile.mkdtemp(prefix="mnist_")
    try:
        full_train = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
        # Reduire pour rapidite : 5000 train, 1000 val, 1000 test
        full_train = Subset(full_train, range(7000))
        train_ds, val_ds = random_split(full_train, [6000, 1000], generator=torch.Generator().manual_seed(SEED))
        test_ds = Subset(datasets.MNIST(data_dir, train=False, download=True, transform=transform), range(1000))
        print(f"Train : {len(train_ds)}, Val : {len(val_ds)}, Test : {len(test_ds)}")
    except Exception as e:
        print(f"MNIST DL failed : {e}\nFallback synthetique...")
        TORCHVISION = False

if not TORCHVISION:
    # Donnees synthetiques (10 classes, images 28x28 random)
    from torch.utils.data import TensorDataset
    n_train, n_val, n_test = 6000, 1000, 1000
    X_full = torch.rand(n_train + n_val + n_test, 1, 28, 28)
    y_full = torch.randint(0, 10, (n_train + n_val + n_test,))
    train_ds = TensorDataset(X_full[:n_train], y_full[:n_train])
    val_ds = TensorDataset(X_full[n_train:n_train+n_val], y_full[n_train:n_train+n_val])
    test_ds = TensorDataset(X_full[n_train+n_val:], y_full[n_train+n_val:])
    print(f"Synthetique : Train={n_train}, Val={n_val}, Test={n_test}")

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=256, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, num_workers=0)

In [ ]:
# === Trainer ===
trainer = L.Trainer(
    max_epochs=1,                   # 1 epoch pour rapidite (demo notebook)
    accelerator="auto",             # CPU / GPU / MPS / TPU auto
    devices=1,                      # 1 device
    log_every_n_steps=20,
    enable_progress_bar=False,      # mode silencieux pour notebook test
    enable_model_summary=False,
    deterministic=True,
)

model = MNISTClassifier(hidden_dim=64, lr=1e-3)
trainer.fit(model, train_loader, val_loader)

# Test final
test_results = trainer.test(model, test_loader, verbose=False)
print(f"\nTest accuracy : {test_results[0]['test_acc']:.4f}")

**Lecture** : Lightning a tout gere — device, train/eval mode, backward/step, validation a chaque epoch.


## 3. Trainer — options principales

```python
trainer = L.Trainer(
    max_epochs=100,
    accelerator="auto",     # cpu / gpu / tpu / mps / auto
    devices=1,              # 1 GPU. "auto" detecte. [0, 2] = GPU 0 et 2
    strategy="ddp",         # distributed sur multi-GPU
    precision="16-mixed",   # FP16 mixed (2× vitesse sur GPU recent)
    # precision="bf16-mixed" pour Ampere+
    # precision="64"        pour float64 (scientifique)
    accumulate_grad_batches=4,   # gradient accumulation (simule batch x4)
    gradient_clip_val=1.0,       # clip gradients pour stabilite
    val_check_interval=0.5,      # validation a mi-epoch
    log_every_n_steps=50,
    deterministic=True,          # reproducibility (slower)
)
```


## 4. Callbacks — extensibilite propre

```python
from lightning.pytorch.callbacks import (
    EarlyStopping, ModelCheckpoint, LearningRateMonitor,
    StochasticWeightAveraging, GradientAccumulationScheduler,
    DeviceStatsMonitor,
)

trainer = L.Trainer(
    max_epochs=100,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, mode="min"),
        ModelCheckpoint(
            monitor="val_acc", mode="max",
            save_top_k=3,
            filename="{epoch}-{val_acc:.3f}",
            dirpath="checkpoints/",
        ),
        LearningRateMonitor(logging_interval="step"),
        StochasticWeightAveraging(swa_lrs=1e-3),    # boost ~ 1-2% gratuit
        DeviceStatsMonitor(),                         # GPU stats dans le log
    ],
)
```

### Callbacks customs
```python
class PrintCallback(L.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        print(f"Epoch {trainer.current_epoch} done, val_loss = {trainer.callback_metrics['val_loss']:.4f}")
```


## 5. Loggers

```python
from lightning.pytorch.loggers import TensorBoardLogger, WandbLogger, MLFlowLogger, CSVLogger

trainer = L.Trainer(
    logger=[
        TensorBoardLogger("tb_logs", name="my_model"),
        WandbLogger(project="my-project"),       # necessite `wandb login`
        MLFlowLogger(experiment_name="exp1"),
        CSVLogger("csv_logs", name="my_model"),
    ],
)
```

Dans le LightningModule : `self.log("metric", value)` route vers tous les loggers configures.


## 6. Mixed precision (FP16 / BF16)

~ 2× rapide sur GPU recent (Ampere+), ~ 50% moins de RAM, perte de precision quasi nulle.

```python
trainer = L.Trainer(precision="16-mixed")    # FP16 mixed (legacy)
trainer = L.Trainer(precision="bf16-mixed")  # BF16 (mieux numeriquement, GPUs Ampere+)
trainer = L.Trainer(precision="32")           # standard FP32
trainer = L.Trainer(precision="64")           # double (scientifique)
```

**Quand l'utiliser** : presque toujours sur GPU recent. Pour les modeles tres deep (Transformers), BF16 > FP16 (range plus large).


## 7. Distributed training

### Single-node multi-GPU (DDP — Distributed Data Parallel)
```python
trainer = L.Trainer(devices=4, strategy="ddp", accelerator="gpu")
```
→ Lance 4 processus, un par GPU. Lightning gere le sync gradient.

### Multi-node
```python
trainer = L.Trainer(devices=4, num_nodes=2, strategy="ddp")  # 8 GPUs total
# Lancer avec : torchrun --nnodes=2 --nproc_per_node=4 --master_addr=... script.py
```

### DeepSpeed / FSDP (modeles tres gros, LLMs)
```python
trainer = L.Trainer(devices=4, strategy="deepspeed_stage_3")  # ZeRO-3 (sharding maximal)
trainer = L.Trainer(devices=4, strategy="fsdp")                # PyTorch native FSDP
```


## 8. LightningDataModule

Pour decoupler la **logique data** du modele. Utile pour reproductibilite et tests.

```python
class MNISTDataModule(L.LightningDataModule):
    def __init__(self, batch_size=128, data_dir="./data"):
        super().__init__()
        self.batch_size = batch_size
        self.data_dir = data_dir

    def prepare_data(self):
        # Telechargements (1 fois global, pas par GPU)
        datasets.MNIST(self.data_dir, train=True, download=True)
        datasets.MNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        # Splits (par GPU)
        transform = transforms.ToTensor()
        if stage == "fit":
            full = datasets.MNIST(self.data_dir, train=True, transform=transform)
            self.train, self.val = random_split(full, [55000, 5000])
        if stage == "test":
            self.test = datasets.MNIST(self.data_dir, train=False, transform=transform)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True, num_workers=4)
    def val_dataloader(self):
        return DataLoader(self.val, batch_size=256, num_workers=4)
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=256, num_workers=4)

# Usage
trainer.fit(model, datamodule=MNISTDataModule())
trainer.test(model, datamodule=MNISTDataModule())
```


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Tout dans `__init__` (data + model + optim) | Separer : `LightningModule` (model), `LightningDataModule` (data) |
| `self.log()` partout en `on_step=True` | Couteux. `on_epoch=True` par defaut |
| `.to(device)` manuel dans le LightningModule | Lightning gere ! |
| `model.eval()` / `model.train()` manuel | Lightning gere ! |
| `loss.backward()` manuel dans `training_step` | Retourner la loss, Lightning fait backward |
| Mixing CPU/GPU manuellement | `accelerator="auto"` |
| Pas de `save_hyperparameters()` | Hparams pas logges → impossible de reproduire |
| Hardcoder le lr | Via `self.hparams.lr` apres `save_hyperparameters()` |
| Oublier `prepare_data` vs `setup` | Prepare = download (1x global), setup = split (par GPU) |
| Pas de `seed_everything` | Reproductibilite cassee |

## 10. References

- **Lightning docs** : https://lightning.ai/docs/pytorch/stable/
- **Lightning Bolts** (templates) : https://lightning-bolts.readthedocs.io/
- **Lightning AI Studio** : https://lightning.ai/  (cloud GPU)
- **TorchMetrics** : https://lightning.ai/docs/torchmetrics/

Voir aussi :
- [DL_PyTorch.ipynb](./DL_PyTorch.ipynb) — PyTorch from-scratch (comprehension)
- [DL_Tensorflow_Keras.ipynb](./DL_Tensorflow_Keras.ipynb) — equivalent Keras
- [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb) — Wandb integration
